In [1]:
from tqdm.notebook import tqdm
from api.API import IslandsAPI
import pandas as pd
import numpy as np
from time import time_ns,sleep

from api.person import Person
from api.tasks import * #Load all tasks
from random import shuffle # Method to randomize groups

tqdm.pandas() #Initialize tqdm


api = IslandsAPI() #Load api
participants = api.get_study_participants() #Get all participants who are contacts
shuffle(participants) #Shuffle the participants list

treatments = {'Control': None, '40% Dark' :CHOCOLATE_DARK_40, '70% Dark' : CHOCOLATE_DARK_70, '85% Dark' : CHOCOLATE_DARK_85, '99% Dark' : CHOCOLATE_DARK_99} #Define treatments as a mapping from labels to tasks

In [3]:
groups = np.array_split(participants, 5) # Split into 5 groups
pd.DataFrame([(person.get_id(), list(treatments.keys())[i]) for (i,group) in enumerate(groups) for person in group], columns=['person_id', 'treatment']).to_csv("treatment_assignments.csv") #Save the assignments as a csv file

TypeError: 'builtin_function_or_method' object is not subscriptable

In [2]:
assignments = pd.read_csv('treatment_assignments.csv',index_col=0) #Read csv in

assignments['person_id'] =  assignments['person_id'].map(api.get_person_manager().get_person) # Map all person ids to actual persons

assignments.head() #Display the first 5 columns

,person_id,treatment
0,Person: Id 8uawxcp4e5,Control
1,Person: Id czwugg2hn6,Control
2,Person: Id sdqs3bgcu9,Control
3,Person: Id m4vtt3nf5y,Control
4,Person: Id slhktb7gtj,Control


In [3]:
def wait_with_progress(endtime_ms : float, description: str):#Create a progress bar until a time is complete
    start = time_ns() // 1_000_000
    total_duration = endtime_ms - start

    with tqdm(total=total_duration, unit="ms", desc=description) as pbar:
        last_elapsed = 0

        while (now := time_ns() // 1_000_000) < endtime_ms:
            elapsed = now - start
            if elapsed > last_elapsed:
                pbar.update(elapsed - last_elapsed)
                last_elapsed = elapsed
            sleep(0.1)

        if last_elapsed < total_duration:
            pbar.update(total_duration - last_elapsed)

In [4]:
tqdm.pandas(desc="Starting Blood Glucose Task")
blood_glucose_results = assignments['person_id'].progress_apply(lambda person: person.do_task(BLOOD_GLUCOSE)).apply(pd.Series) #Have every person measure their blood glucose levels

wait_with_progress(blood_glucose_results['end_time'].max(), 'Waiting for Blood Glucose Results')

tqdm.pandas(desc='Retrieving Blood Glucose Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())
assignments['base_blood_glucose'] = assignments['person_id'].apply(lambda p: p.get_task_results()[0].result())
assignments.head()

Starting Blood Glucose Task:   0%|          | 0/62 [00:00<?, ?it/s]

Waiting for Blood Glucose Results:   0%|          | 0/23090.0 [00:00<?, ?ms/s]

Retrieving Blood Glucose Results:   0%|          | 0/62 [00:00<?, ?it/s]

,person_id,treatment,base_blood_glucose
0,Person: Id 8uawxcp4e5,Control,92 mg/dL
1,Person: Id czwugg2hn6,Control,90 mg/dL
2,Person: Id sdqs3bgcu9,Control,85 mg/dL
3,Person: Id m4vtt3nf5y,Control,87 mg/dL
4,Person: Id slhktb7gtj,Control,96 mg/dL


In [10]:
def run_experiment(row):
    if row['treatment'] == 'Control':
        return None
    return row['person_id'].do_task(treatments[row['treatment']])

tqdm.pandas(desc='Starting Eat Chocolate Task')
chocolate_times = assignments.progress_apply(run_experiment,axis=1).apply(pd.Series)
wait_with_progress(chocolate_times['end_time'].max(), "Eating Chocolate")

tqdm.pandas(desc="Starting Blood Glucose Task")
blood_glucose_results = assignments['person_id'].progress_apply(lambda person: person.do_task(BLOOD_GLUCOSE)).apply(pd.Series) #Have every person measure their blood glucose levels

wait_with_progress(blood_glucose_results['end_time'].max(), 'Waiting for Blood Glucose Results')

tqdm.pandas(desc='Retrieving Blood Glucose Results')
assignments['person_id'].progress_apply(lambda p: p.update_person())
assignments['end_blood_glucose'] = assignments['person_id'].apply(lambda p: p.get_task_results()[0].result())

Starting Eat Chocolate Task:   0%|          | 0/62 [00:00<?, ?it/s]

Eating Chocolate:   0%|          | 0/140000.10009765625 [00:00<?, ?ms/s]

Starting Blood Glucose Task:   0%|          | 0/62 [00:00<?, ?it/s]

Waiting for Blood Glucose Results:   0%|          | 0/23242.0 [00:00<?, ?ms/s]

Retrieving Blood Glucose Results:   0%|          | 0/62 [00:00<?, ?it/s]

In [29]:
assignments['end_blood_glucose'] =  assignments['end_blood_glucose'].str.split(' ').map(lambda x: x[0]).astype(int) #Clean up rows so blood glucose is an integer
assignments['base_blood_glucose'] =  assignments['base_blood_glucose'].str.split(' ').map(lambda x: x[0]).astype(int) #Clean up rows so blood glucose is an integer
assignments.to_csv("chocolate_experiment_results.csv")

0     89
1     88
2     87
3     84
4     96
      ..
57    88
58    89
59    92
60    91
61    92
Name: end_blood_glucose, Length: 62, dtype: int64